# 3 — Metric learning as a defence

Circle Loss (Sun et al., 2020) reshapes the penultimate space rather than the
decision rule: pull same-class embeddings together, push different-class
embeddings apart, and weight each pair by how far it currently is from where
it should be.

$$L = \log\Big[1 + \sum_i \exp\big(\gamma\alpha^+_i(s^+_i - \Delta)\big)
+ \sum_j \exp\big(\gamma\alpha^-_j(\Delta - s^-_j)\big)\Big]$$

Two variants:

- **Pair-based** — similarities are computed between embeddings inside the
  batch, so what the loss sees depends on how the batch was sampled.
- **Proxy-based** — one learned vector per class, similarities computed
  against those. One matrix product per batch, no pair mining, and the proxies
  double as a classifier.

Neither variant ever sees an adversarial image during training. The question
is whether a better-shaped embedding is robust by itself.

`advlab` defines every attack on images in `[0, 1]`, so `eps=8/255` is eight
levels of an 8-bit pixel — the standard CIFAR-10 L-infinity budget — and it
means the same thing in all four notebooks.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import advlab
from advlab import CLASSES, DEVICE, accuracy, apgd, cifar10, fgsm, mifgsm, pgd

advlab.set_seed()
print("device:", DEVICE)

In [ ]:
from pytorch_metric_learning.losses import CircleLoss

EPS = 8 / 255
BATCH = 128
EMBED_DIM = 64          # the penultimate width of ResNet-20

test_loader = cifar10(train=False, batch_size=256)
train_loader = cifar10(train=True, batch_size=BATCH, fraction=0.5)


def report(model, label):
    a_clean = accuracy(model, test_loader)
    a_fgsm = accuracy(model, test_loader, attack=fgsm, eps=EPS)
    a_pgd = accuracy(model, test_loader, attack=pgd, eps=EPS,
                     alpha=2 / 255, steps=10)
    print(f"{label:24s} clean {a_clean:6.2f}%   FGSM {a_fgsm:6.2f}%"
          f"   PGD {a_pgd:6.2f}%")
    return a_clean, a_fgsm, a_pgd

## Pair-based Circle Loss

Two stages, because the loss shapes an embedding and does not produce logits:
40 epochs on the embedding, then 10 epochs training a fresh linear head on the
frozen features.

Stage one needs the network to *return* penultimate features with a gradient
attached, which the `Penultimate` pre-hook used for plotting cannot do — it
detaches. So the head is swapped for an `Identity`, inside a context manager
that always puts the real head back. The swap must be undone before anything
calls `CrossEntropyLoss`: with the head still an `Identity`, cross-entropy
would be comparing labels `0-9` against a 64-dimensional vector as if it were
64 class logits, and `argmax` over it would be a 64-way decision.

In [ ]:
import contextlib


@contextlib.contextmanager
def as_embedder(model):
    """Temporarily make `model` return penultimate features, with gradient."""
    head = model.model.fc
    model.model.fc = nn.Identity()
    try:
        yield model
    finally:
        model.model.fc = head


EPOCHS_EMBED = 40

advlab.set_seed()
circle_model = advlab.resnet20(pretrained=True)
circle_loss = CircleLoss(m=0.25, gamma=256).to(DEVICE)
opt = torch.optim.Adam(circle_model.parameters(), lr=1e-4)

embed_history = []
with as_embedder(circle_model) as embedder:
    for epoch in range(EPOCHS_EMBED):
        embedder.train()
        running = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad()
            loss = circle_loss(embedder(images), labels)
            loss.backward()
            opt.step()
            running += loss.item()
        embed_history.append(running / len(train_loader))
        if epoch % 5 == 0 or epoch == EPOCHS_EMBED - 1:
            print(f"embedding epoch {epoch + 1:2d}/{EPOCHS_EMBED}"
                  f"  circle loss {embed_history[-1]:.4f}")

In [ ]:
EPOCHS_HEAD = 10

# Fresh head on the reshaped embedding; the body stays frozen.
circle_model.model.fc = nn.Linear(EMBED_DIM, 10).to(DEVICE)
for name, param in circle_model.named_parameters():
    param.requires_grad = name.startswith("model.fc")

opt = torch.optim.Adam(circle_model.model.fc.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

head_history = []
for epoch in range(EPOCHS_HEAD):
    circle_model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        opt.zero_grad()
        loss = loss_fn(circle_model(images), labels)
        loss.backward()
        opt.step()
        running += loss.item()
    head_history.append(running / len(train_loader))
    print(f"head epoch {epoch + 1:2d}/{EPOCHS_HEAD}"
          f"  cross-entropy {head_history[-1]:.4f}")

for param in circle_model.parameters():
    param.requires_grad = True

In [ ]:
circle_scores = report(circle_model, "Circle Loss (pair)")

# Two stages of different lengths and different units: separate axes.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(range(1, EPOCHS_EMBED + 1), embed_history, marker="o", ms=3)
ax1.set(xlabel="epoch", ylabel="circle loss", title="Stage 1 — embedding")
ax1.grid(alpha=0.3)
ax2.plot(range(1, EPOCHS_HEAD + 1), head_history, marker="o", color="tab:orange")
ax2.set(xlabel="epoch", ylabel="cross-entropy", title="Stage 2 — linear head")
ax2.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
feats, labels, conds = advlab.collect_features(
    circle_model, test_loader, n_samples=1000, eps=EPS)
advlab.plot_umap(feats, labels, conds, "Penultimate features, Circle Loss")
plt.show()

## Proxy-based Circle Loss

One learned proxy per class, cosine similarity between each embedding and each
proxy, Circle Loss on those similarities. The proxies are also the classifier:
`argmax` over the similarities is the prediction, so there is no second stage.

In [ ]:
class ProxyHead(nn.Module):
    """Cosine similarity between an embedding and one proxy per class."""

    def __init__(self, dim=EMBED_DIM, n_classes=10, scale=16.0):
        super().__init__()
        self.proxies = nn.Parameter(torch.randn(n_classes, dim) * 0.01)
        self.scale = scale

    def forward(self, e):
        return self.scale * F.normalize(e, dim=1) @ F.normalize(
            self.proxies, dim=1).t()


advlab.set_seed()
proxy_model = advlab.resnet20(pretrained=True)
proxy_model.model.fc = ProxyHead().to(DEVICE)

circle_loss = CircleLoss(m=0.25, gamma=256).to(DEVICE)
opt = torch.optim.Adam(proxy_model.parameters(), lr=1e-4)

EPOCHS_PROXY = 40
proxy_history = []
for epoch in range(EPOCHS_PROXY):
    proxy_model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        opt.zero_grad()
        loss = circle_loss(proxy_model(images), labels)
        loss.backward()
        opt.step()
        running += loss.item()
    proxy_history.append(running / len(train_loader))
    if epoch % 5 == 0 or epoch == EPOCHS_PROXY - 1:
        print(f"epoch {epoch + 1:2d}/{EPOCHS_PROXY}"
              f"  proxy circle loss {proxy_history[-1]:.4f}")

In [ ]:
proxy_scores = report(proxy_model, "Circle Loss (proxy)")

plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS_PROXY + 1), proxy_history, marker="o", ms=3)
plt.xlabel("epoch"), plt.ylabel("proxy circle loss")
plt.title("Proxy-based Circle Loss: training loss")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
feats, labels, conds = advlab.collect_features(
    proxy_model, test_loader, n_samples=1000, eps=EPS)
advlab.plot_umap(feats, labels, conds,
                 "Penultimate features, proxy Circle Loss")
plt.show()

## Comparison

In [ ]:
baseline = advlab.resnet20(pretrained=True)
base_scores = report(baseline, "no defence")
print()
print(f"{'model':24s} {'clean':>8s} {'FGSM':>8s} {'PGD':>8s}")
for label, s in [("no defence", base_scores),
                 ("Circle Loss (pair)", circle_scores),
                 ("Circle Loss (proxy)", proxy_scores)]:
    print(f"{label:24s} {s[0]:7.2f}% {s[1]:7.2f}% {s[2]:7.2f}%")

Both variants keep nearly all the clean accuracy and give some robustness, and
the proxy version is clearly the better of the two — pair-based Circle Loss
depends on what pairs happen to land in a batch, while a proxy is a stable
target every batch updates.

But neither is a defence in the sense adversarial training is, and the reason
is structural: Circle Loss optimises distances between *clean* embeddings. It
makes same-class images cluster and different-class clusters separate, which
is worth having, but nothing in the objective says the embedding must vary
slowly as the input moves inside an epsilon ball. That is the property
robustness actually needs, and it is not what is being optimised. Metric
learning is a sensible regulariser to combine with adversarial training, not a
replacement for it.